## LSTM

## 1. Importing Libraries

In [2]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

TensorFlow version: 2.21.0
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [3]:
DATA_PATH = "daily_commodity_market_data_cleaned.csv"
TARGET_COLUMN = "tomorrow_gold_log_return"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Date"],
    index_col="Date",
).sort_index()

print("Shape:", df.shape)
print("Start date:", df.index.min())
print("End date:", df.index.max())
print("Duplicate dates:", df.index.duplicated().sum())
print("Missing values:", df.isna().sum().sum())
print("Target present:", TARGET_COLUMN in df.columns)

df.head()

Shape: (5016, 22)
Start date: 2004-01-22 00:00:00
End date: 2026-07-13 00:00:00
Duplicate dates: 0
Missing values: 0
Target present: True


,tomorrow_gold_log_return,gold_return_1d,gold_return_5d,gold_return_20d,gold_volatility_5d,gold_volatility_20d,gold_close_pos,gold_volume_change,gold_relative_volume,crude_oil_return_1d,...,platinum_return_1d,us_dollar_index_return_1d,sp_500_return_1d,eur_usd_return_1d,vix_change_1d,breakeven_inflation_10y_diff_bps,us_2_year_treasury_yields_diff_bps,us_10_year_treasury_yields_diff_bps,two_ten_slope,two_ten_slope_change_bps
Date,,,,,,,,,,,,,,,,,,,,,
2004-01-22,-0.005135,-0.002679,-0.028137,-0.000975,0.016754,0.010122,0.5,0.000000,0.060060,0.010071,...,0.011614,-0.006273,-0.003212,0.005204,0.37,-3.0,-3.0,-6.0,2.33,-3.0
2004-01-23,-0.003192,-0.005135,-0.001225,-0.003182,0.008370,0.010167,0.5,0.000000,0.069686,0.000286,...,0.002056,0.008586,-0.002091,-0.011224,0.13,4.0,5.0,10.0,2.38,5.0
2004-01-26,0.008327,-0.003192,-0.000246,-0.009790,0.008266,0.010153,0.5,0.000000,0.076336,-0.013543,...,-0.018656,0.007940,0.012034,-0.007584,-0.29,0.0,4.0,7.0,2.41,3.0
2004-01-27,0.010916,0.008327,-0.006564,-0.002436,0.005467,0.010340,0.5,0.000000,0.117302,-0.010206,...,0.010177,-0.011180,-0.009846,0.011859,0.80,0.0,-5.0,-5.0,2.41,0.0
2004-01-28,-0.039365,0.010916,0.008236,0.005322,0.007394,0.010611,0.5,10.146747,19.954106,-0.014763,...,-0.022104,0.007965,-0.013703,-0.015548,1.43,3.0,17.0,11.0,2.35,-6.0
